# Narrador IA — ligar o servidor (GPU)

Este notebook só liga o "motor" que roda no Colab. Depois de rodar, você recebe um link — a partir daí, **tudo acontece na página web** (escolher voz, ouvir prévia, colar o roteiro, gerar e baixar). Não precisa mais voltar aqui, exceto pra religar se a sessão do Colab cair.

**Antes de rodar:** menu **Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)** → Salvar.

**Depois:** menu **Ambiente de execução → Executar tudo** (ou Ctrl+F9).

In [ ]:
import os
import subprocess

sPastaProjeto = "/content/narrador"

if os.path.isdir(sPastaProjeto):
    # reset --hard (em vez de pull) porque o histórico remoto pode ter sido
    # reescrito (force-push) desde o último clone nesta sessão do Colab —
    # pull falharia tentando reconciliar históricos sem ancestral comum.
    subprocess.run(["git", "-C", sPastaProjeto, "fetch", "origin"], check=True)
    subprocess.run(
        ["git", "-C", sPastaProjeto, "reset", "--hard", "origin/prod"], check=True
    )
    subprocess.run(["git", "-C", sPastaProjeto, "clean", "-fd"], check=True)
else:
    subprocess.run(
        ["git", "clone", "https://github.com/guzsysdev/narrador.git", sPastaProjeto],
        check=True,
    )

os.chdir(sPastaProjeto)

# O Colab já vem com PyTorch + CUDA — não reinstalamos torch para não quebrar isso.
subprocess.run(
    ["pip", "install", "-q", "voxcpm", "librosa", "soundfile",
     "fastapi", "uvicorn", "pydantic", "python-multipart", "pyngrok"],
    check=True,
)

import torch
print("GPU disponível:", torch.cuda.is_available())
print("Dispositivo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — ative GPU no menu Ambiente de execução")

### Token do ngrok (só a primeira vez)

Pra expor o servidor numa URL pública, usamos o ngrok (grátis). Pra nunca mais ter que colar o token:

1. Crie uma conta grátis em https://ngrok.com e pegue seu authtoken em https://dashboard.ngrok.com/get-started/your-authtoken
2. No Colab, clique no ícone de **chave 🔑** na barra lateral esquerda ("Secrets")
3. Adicione um novo secret: nome `NGROK_TOKEN`, valor = seu authtoken
4. Ative a chavinha "Acesso ao notebook" pra esse secret

Feito isso uma vez, a célula abaixo encontra o token sozinha em qualquer sessão futura. Se não configurar, ela vai pedir pra colar manualmente (também funciona, só que toda vez).

In [ ]:
sTokenNgrok = None
try:
    from google.colab import userdata
    sTokenNgrok = userdata.get("NGROK_TOKEN")
except Exception:
    pass

if not sTokenNgrok:
    from getpass import getpass
    sTokenNgrok = getpass("Secret NGROK_TOKEN não encontrado — cole seu ngrok authtoken: ")

In [ ]:
import time
import json as jsonlib
import urllib.request
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokHTTPError

ngrok.set_auth_token(sTokenNgrok)
ngrok.kill()  # encerra qualquer túnel LOCAL desta sessão (não afeta outra sessão do Colab)

oProcessoServidor = subprocess.Popen(
    ["uvicorn", "servidorWeb:oApp", "--host", "0.0.0.0", "--port", "8000"]
)

print("Aguardando o modelo carregar (alguns minutos na primeira vez)...", end="", flush=True)
iTentativas = 0
iTentativasMaximas = 180  # ~15 min (180 x 5s) — depois disso, algo está errado
while iTentativas < iTentativasMaximas:
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/api/saude", timeout=3) as oResp:
            oStatus = jsonlib.loads(oResp.read())
            if oStatus.get("sErro"):
                print()
                raise RuntimeError(f"O modelo falhou ao carregar: {oStatus['sErro']}")
            if oStatus.get("lPronto"):
                break
    except urllib.error.URLError:
        pass  # servidor ainda subindo, tenta de novo
    print(".", end="", flush=True)
    time.sleep(5)
    iTentativas += 1
else:
    raise TimeoutError(
        "Modelo não ficou pronto em 15 min. Role para cima nesta célula e procure "
        "por um traceback no log do uvicorn (pode ter travado sem erro explícito)."
    )
print()

# ERR_NGROK_334 ("endpoint already online") acontece quando outra sessão do Colab
# (aba antiga, ainda aberta ou não totalmente encerrada) está com esse mesmo túnel
# ativo — o plano grátis do ngrok só permite 1 túnel por vez. Tentamos de novo
# algumas vezes (a sessão antiga costuma cair sozinha por timeout de inatividade
# em alguns segundos/minutos); se persistir, é preciso encerrar a outra sessão.
oTunel = None
for iTentativaTunel in range(5):
    try:
        oTunel = ngrok.connect(8000)
        break
    except PyngrokNgrokHTTPError as oErro:
        if "already online" not in str(oErro):
            raise
        print(f"Túnel já em uso por outra sessão do Colab, tentando de novo em 15s "
              f"({iTentativaTunel + 1}/5)...")
        time.sleep(15)

if oTunel is None:
    raise RuntimeError(
        "Não consegui abrir o túnel: outra sessão do Colab ainda está usando o mesmo "
        "link do ngrok. Feche/encerre qualquer outra aba do Colab com este notebook "
        "aberto (Ambiente de execução → Encerrar sessão nela), ou pare o túnel manualmente "
        "em https://dashboard.ngrok.com/agents — depois rode esta célula de novo."
    )

print("=" * 60)
print("Modelo pronto! Abra esta URL no navegador:", oTunel.public_url)
print("=" * 60)

# Mantém esta célula "ocupada" de propósito: o Colab desconecta por
# inatividade (~90 min) quando nenhuma célula está executando — e sem
# isso, a célula terminaria assim que imprimisse a URL acima, mesmo com
# o servidor (uvicorn) ainda rodando em background. Este loop finge
# atividade pra manter o kernel "ocupado" de verdade.
#
# Isso NÃO evita o limite absoluto de sessão do plano grátis (~12h) —
# só evita a desconexão por ociosidade. Pra sessões mais longas/em
# background de verdade, só o Colab Pro/Pro+ resolve.
#
# Pra encerrar: clique no botão de stop desta célula (não afeta o
# servidor, que continua rodando — só pare a sessão de vez pelo menu
# Ambiente de execução → Encerrar sessão quando quiser desligar tudo).
print()
print("Servidor ativo — mantendo a sessão viva (botão de stop pra interromper "
      "este aviso, sem derrubar o servidor).")
iMinutosAtivo = 0
while True:
    time.sleep(60)
    iMinutosAtivo += 1
    print(f"[{iMinutosAtivo} min] servidor ativo — {oTunel.public_url}")

## Pronto

Abra o link acima no navegador — é lá que você faz tudo: escolhe a voz, ouve a prévia, cola o roteiro completo, ajusta velocidade/tom, gera e baixa o WAV final.

A URL muda toda vez que a célula acima é executada de novo — não compartilhe, qualquer pessoa com o link consegue usar o servidor enquanto ele estiver no ar.

Se a sessão do Colab cair (limite do plano grátis), volte aqui e rode **Ambiente de execução → Executar tudo** de novo — em ~1-2 min sai um novo link.

### Pra encerrar o servidor mais tarde

NÃO adicione uma célula de `ngrok.disconnect()`/`terminate()` aqui — se você usar "Executar tudo" de novo, ela rodaria logo em seguida e derrubaria o servidor que acabou de subir. Pra encerrar, use o menu **Ambiente de execução → Encerrar sessão** (ou **Reiniciar sessão** se quiser rodar de novo em seguida) — isso mata o servidor e o túnel de forma limpa, sem risco de conflito com "Executar tudo".